In [ ]:
### ANTS Registration GUI for Jupyter Notebook
This notebook provides a single, easy-to-use GUI tool for image registration using the ANTS library. The tool is designed to help you select the best-performing registration parameters based on pre-calculated values and apply them to your images.

How to Use
Dependencies: Before running the code, ensure you have the required libraries installed in your environment: ants, tifffile, pandas, and tkinter. You can install them using pip: pip install ants tifffile pandas. tkinter is a standard Python library.

Run the Cell: Execute the entire code block in a Jupyter notebook cell. This will launch a Tkinter window for interactive use.

Use the GUI:

Image File Selection: Use the "Select" buttons to choose your Fixed Image (the target for registration), Reference Image (the moving image that defines the transformation), and any other channels you want to transform using the same parameters.

Registration Parameter Selection:

The GUI is pre-configured with the top 5 best-performing parameter sets based on your previous analysis. Select one of these from the dropdown menu to automatically populate the parameter fields.

Alternatively, choose "Manual Input" to enter your own custom values for Metric, Sampling, and Iterations.

Execution: Click the "Run" button to start the registration process. Progress and results will be displayed in the log text box within the GUI.

Results: After the process completes, the registered images (.tif and .nii.gz), a CSV file containing deformation vectors, and a log of the process will be saved to a new folder named registration_results in the same directory as your fixed image.



In [6]:
# -*- coding: utf-8 -*-
"""
250626 -- Aiming to surpass Pattern 6 with 50 different parameters (#05-#54)
Input directory: "~/Downloads" -- Same as previous code.
"""

import os
import shutil
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import ants
import tifffile
import numpy as np
import pandas as pd
import re

# ────────────────────────────────
# 1) Parameter Definition and GUI
# ────────────────────────────────
class RegistrationGUI:
    def __init__(self, master):
        self.master = master
        master.title("ANTS Registration Tool")
        master.geometry("800x600")

        self.fixed_path = None
        self.ref_path = None
        self.other_paths = {}
        # Best 5 patterns based on NCC_summary.csv (highest correlation coefficient), re-labeled
        self.best_patterns = [
            ("01_CC_s2_i350-240-160-80", "CC", 2, [350, 240, 160, 80]),
            ("02_CC_s3_i350-240-160-80", "CC", 3, [350, 240, 160, 80]),
            ("03_CC_s4_i260-180-120-60", "CC", 4, [260, 180, 120, 60]),
            ("04_CC_s2_i320-240-160-80", "CC", 2, [320, 240, 160, 80]),
            ("05_MI_s3_i280-200-150-100", "MI", 3, [280, 200, 150, 100]),
        ]

        self.create_widgets()

    def create_widgets(self):
        # Frame for file selection
        file_frame = ttk.LabelFrame(self.master, text="Image File Selection")
        file_frame.pack(padx=10, pady=10, fill="x")

        ttk.Label(file_frame, text="Fixed Image:").grid(row=0, column=0, sticky="w", padx=5, pady=5)
        self.fixed_path_label = ttk.Label(file_frame, text="Not selected")
        self.fixed_path_label.grid(row=0, column=1, sticky="w", padx=5, pady=5)
        ttk.Button(file_frame, text="Select", command=self.select_fixed).grid(row=0, column=2, padx=5, pady=5)

        ttk.Label(file_frame, text="Reference Image:").grid(row=1, column=0, sticky="w", padx=5, pady=5)
        self.ref_path_label = ttk.Label(file_frame, text="Not selected")
        self.ref_path_label.grid(row=1, column=1, sticky="w", padx=5, pady=5)
        ttk.Button(file_frame, text="Select", command=self.select_ref).grid(row=1, column=2, padx=5, pady=5)

        ttk.Label(file_frame, text="Other Channels:").grid(row=2, column=0, sticky="w", padx=5, pady=5)
        self.other_paths_label = ttk.Label(file_frame, text="Not selected")
        self.other_paths_label.grid(row=2, column=1, sticky="w", padx=5, pady=5)
        ttk.Button(file_frame, text="Select", command=self.select_others).grid(row=2, column=2, padx=5, pady=5)

        # Frame for parameter selection
        param_frame = ttk.LabelFrame(self.master, text="Registration Parameter Selection")
        param_frame.pack(padx=10, pady=10, fill="x")

        ttk.Label(param_frame, text="Pattern Options:").grid(row=0, column=0, sticky="w", padx=5, pady=5)
        self.pattern_combo = ttk.Combobox(param_frame, state="readonly", width=50)
        self.pattern_combo.grid(row=0, column=1, columnspan=2, padx=5, pady=5)
        self.pattern_combo.bind("<<ComboboxSelected>>", self.on_pattern_select)

        ttk.Label(param_frame, text="Or Manual Input:").grid(row=1, column=0, sticky="w", padx=5, pady=5)
        
        ttk.Label(param_frame, text="Metric (CC/MI):").grid(row=2, column=0, sticky="w", padx=5, pady=5)
        self.metric_entry = ttk.Entry(param_frame)
        self.metric_entry.grid(row=2, column=1, sticky="w", padx=5, pady=5)

        ttk.Label(param_frame, text="Sampling (1-4):").grid(row=3, column=0, sticky="w", padx=5, pady=5)
        self.samp_entry = ttk.Entry(param_frame)
        self.samp_entry.grid(row=3, column=1, sticky="w", padx=5, pady=5)

        ttk.Label(param_frame, text="Iterations (e.g., 200-140-80):").grid(row=4, column=0, sticky="w", padx=5, pady=5)
        self.iters_entry = ttk.Entry(param_frame)
        self.iters_entry.grid(row=4, column=1, sticky="w", padx=5, pady=5)
        
        # Frame for execution and log
        exec_frame = ttk.LabelFrame(self.master, text="Execution")
        exec_frame.pack(padx=10, pady=10, fill="x")

        self.run_button = ttk.Button(exec_frame, text="Run", command=self.run_registration)
        self.run_button.pack(pady=10)
        
        self.log_text = tk.Text(self.master, height=15, state="disabled")
        self.log_text.pack(padx=10, pady=10, fill="both", expand=True)
        
        # Initialize the combobox with hardcoded best patterns
        pattern_labels = [p[0] for p in self.best_patterns]
        self.pattern_combo['values'] = ["Manual Input"] + pattern_labels
        self.pattern_combo.set(pattern_labels[0]) # Set the best pattern as default
        self.on_pattern_select(None) # Call to populate input fields

    def log(self, message):
        self.log_text.configure(state="normal")
        self.log_text.insert(tk.END, message + "\n")
        self.log_text.see(tk.END)
        self.log_text.configure(state="disabled")

    def select_fixed(self):
        path = filedialog.askopenfilename(defaultextension=".tif", filetypes=[("TIFF files", "*.tif")])
        if path:
            self.fixed_path = path
            self.fixed_path_label.config(text=os.path.basename(path))

    def select_ref(self):
        path = filedialog.askopenfilename(defaultextension=".tif", filetypes=[("TIFF files", "*.tif")])
        if path:
            self.ref_path = path
            self.ref_path_label.config(text=os.path.basename(path))

    def select_others(self):
        paths = filedialog.askopenfilenames(defaultextension=".tif", filetypes=[("TIFF files", "*.tif")])
        if paths:
            self.other_paths = {os.path.splitext(os.path.basename(p))[0].split('_')[-2]: p for p in paths}
            self.other_paths_label.config(text=", ".join(self.other_paths.keys()))

    def on_pattern_select(self, event):
        selected_label = self.pattern_combo.get()
        if selected_label == "Manual Input":
            self.metric_entry.delete(0, tk.END)
            self.samp_entry.delete(0, tk.END)
            self.iters_entry.delete(0, tk.END)
        else:
            for label, metric, samp, iters in self.best_patterns:
                if label == selected_label:
                    self.metric_entry.delete(0, tk.END)
                    self.metric_entry.insert(0, metric)
                    self.samp_entry.delete(0, tk.END)
                    self.samp_entry.insert(0, str(samp))
                    self.iters_entry.delete(0, tk.END)
                    self.iters_entry.insert(0, '-'.join(map(str, iters)))
                    break

    def run_registration(self):
        self.log_text.delete(1.0, tk.END)
        self.log("--- Starting Execution ---")

        if not self.fixed_path or not self.ref_path:
            messagebox.showerror("Error", "Please select both a fixed and a reference image.")
            self.log("Execution failed: Files not selected.")
            return

        try:
            metric = self.metric_entry.get().strip()
            samp = int(self.samp_entry.get().strip())
            iters_str = self.iters_entry.get().strip()
            iters = [int(i) for i in iters_str.split('-')]
            
            if not metric or not samp or not iters:
                messagebox.showerror("Error", "Incomplete parameters.")
                self.log("Execution failed: Incomplete parameters.")
                return

            self.log(f"Parameters: Metric={metric}, Sampling={samp}, Iterations={iters}")

            outroot = os.path.join(os.path.dirname(self.fixed_path), "registration_results")
            os.makedirs(outroot, exist_ok=True)
            tag = f"{metric}_s{samp}_i{iters_str}"
            odir = os.path.join(outroot, tag)
            os.makedirs(odir, exist_ok=True)

            self.log("Loading images...")
            fixed_img = ants.image_read(self.fixed_path)
            ref_img = ants.image_read(self.ref_path)

            self.log("Starting image registration...")
            reg = ants.registration(
                fixed=fixed_img,
                moving=ref_img,
                type_of_transform="SyN",
                syn_metric=metric,
                syn_sampling=samp,
                syn_iterations=iters,
                verbose=False,
            )
            self.log("Image registration completed.")

            fwd_paths = []
            for i, tf in enumerate(reg["fwdtransforms"]):
                dst = os.path.join(odir, f"{tag}_fwd_{i}_{os.path.basename(tf)}")
                shutil.copy(tf, dst)
                fwd_paths.append(dst)

            warped = reg["warpedmovout"]
            warped.to_file(os.path.join(odir, f"{tag}_warped.nii.gz"))
            tifffile.imwrite(
                os.path.join(odir, f"{tag}_warped.tif"),
                warped.numpy().transpose(2, 1, 0).astype(np.uint16),
            )
            mi, cc = self.evaluate_similarity(fixed_img, warped)
            self.log(f"POPO1: MI={mi:.4f}, CC={cc:.4f}")

            # deformation vectors
            vec = ants.image_read(reg["fwdtransforms"][0]).numpy().reshape(3, -1).T
            pd.DataFrame(vec, columns=["dx", "dy", "dz"]).to_csv(
                os.path.join(odir, f"{tag}_deformation_vectors.csv"), index=False
            )

            # other channels
            for label, mp in self.other_paths.items():
                self.log(f"Applying transforms to {label}...")
                warped_other = ants.apply_transforms(
                    fixed=fixed_img,
                    moving=ants.image_read(mp),
                    transformlist=fwd_paths,
                )
                warped_other.to_file(os.path.join(odir, f"{tag}_warped_{label}.nii.gz"))
                tifffile.imwrite(
                    os.path.join(odir, f"{tag}_warped_{label}.tif"),
                    warped_other.numpy().transpose(2, 1, 0).astype(np.uint16),
                )
                mi, cc = self.evaluate_similarity(fixed_img, warped_other)
                self.log(f"{label}: MI={mi:.4f}, CC={cc:.4f}")

            self.log("--- Execution Completed ---")

        except Exception as e:
            messagebox.showerror("Error", f"An error occurred during execution: {e}")
            self.log(f"Error: {e}")

    def evaluate_similarity(self, fixed_img, moving_img):
        mi = ants.image_mutual_information(fixed_img, moving_img)
        f, m = fixed_img.numpy().astype(np.float32), moving_img.numpy().astype(np.float32)
        f = (f - f.mean()) / f.std()
        m = (m - m.mean()) / m.std()
        cc = np.corrcoef(f.ravel(), m.ravel())[0, 1]
        return mi, cc

if __name__ == "__main__":
    root = tk.Tk()
    app = RegistrationGUI(root)
    root.mainloop()